In [3]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [4]:
# Local example
BASE_PATH = Path("/Users/shen/Documents/Research/kaggle/jane/train.parquet")

# Kaggle example
# BASE_PATH = Path("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet")

FEATURE_COLS = [f"feature_{i:02d}" for i in range(79)]
TARGET_COL = "responder_6"
WEIGHT_COL = "weight"
DATE_COL = "date_id"
TIME_COL = "time_id"
SYMBOL_COL = "symbol_id"

# Default tuning / backtest settings
DEFAULT_TUNE_SAMPLE_FRAC = 0.35
DEFAULT_PRED_CLIP = None   # example: (-5, 5)
DEFAULT_RANDOM_SEED = 42

In [5]:
def official_weighted_r2(y_true, y_pred, sample_weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = np.asarray(sample_weight, dtype=float)

    ss_res = np.sum(w * (y_true - y_pred) ** 2)
    ss_tot = np.sum(w * y_true ** 2)

    return 1.0 - ss_res / ss_tot

In [6]:
# def sample_train_by_group(df, sample_frac=0.2, group_cols=["date_id"], seed=42,):
#     sampled = (
#         df.group_by(*group_cols, maintain_order=True)
#           .map_groups(
#               lambda g: g.sample(
#                   n=max(1, int(np.ceil(g.height * sample_frac))),
#                   seed=seed
#               )
#           )
#     )

#     return sampled


def load_partitions(partition_ids, base_path=BASE_PATH, sample_frac=0.2, seed=42):

    dfs = []
    for pid in partition_ids:
        #print(f"Loading partition {pid} ...")
        part_path = base_path / f"partition_id={pid}" / "part-0.parquet"
        df = pl.read_parquet(part_path)
        df = sample_train_by_group(df,
            sample_frac=sample_frac, 
            group_cols=[DATE_COL, TIME_COL],
            seed=seed + int(pid),
        )
        dfs.append(df)

    df_all = pl.concat(dfs, how="vertical_relaxed")
    return df_all

In [7]:
def sample_train_by_group(df, sample_frac=0.2, group_cols=["date_id", "time_id"], seed=42):

    if sample_frac >= 1.0 or sample_frac is None:
        return df

    is_polars = isinstance(df, pl.DataFrame)
    if is_polars:
        pdf = df.to_pandas()
    else:
        pdf = df.copy()

    rng = np.random.default_rng(seed)
    pdf['__rand__'] = rng.random(len(pdf))

    group_sizes = pdf.groupby(group_cols)['__rand__'].transform('size')
    target_n = np.maximum(1, np.floor(group_sizes * sample_frac))

    pdf['__rank__'] = pdf.groupby(group_cols)['__rand__'].rank(method='first')
    sampled_pdf = pdf[pdf['__rank__'] <= target_n].drop(columns=['__rand__', '__rank__'])

    if is_polars:
        return pl.from_pandas(sampled_pdf)
    else:
        return sampled_pdf

In [8]:
# def prepare_xywd(
#     df,
#     feature_cols=FEATURE_COLS,
#     target_col=TARGET_COL,
#     weight_col=WEIGHT_COL,
#     date_col=DATE_COL,
#     symbol_col=SYMBOL_COL,
# ):

#     df = df.filter(pl.col(target_col).is_not_null())
#     numeric_exprs = [pl.col(c).cast(pl.Float32).fill_null(float("nan")).alias(c) for c in feature_cols]
#     symbol_expr = pl.col(symbol_col).fill_null(-1).cast(pl.Int32).alias(symbol_col)
#     X = df.select([*numeric_exprs, symbol_expr])
#     y = df.get_column(target_col).cast(pl.Float32).to_numpy()
#     w = df.get_column(weight_col).fill_null(0.0).cast(pl.Float32).to_numpy()
#     d = df.get_column(date_col).to_numpy()

#     return X, y, w, d

In [9]:
def prepare_xywd(
    df,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    weight_col=WEIGHT_COL,
    date_col=DATE_COL,
):

    df = df.filter(pl.col(target_col).is_not_null())
    numeric_exprs = [pl.col(c).cast(pl.Float32).fill_null(float("nan")).alias(c) for c in feature_cols]
    X = df.select(numeric_exprs)
    y = df.get_column(target_col).cast(pl.Float32).to_numpy()
    w = df.get_column(weight_col).fill_null(0.0).cast(pl.Float32).to_numpy()
    d = df.get_column(date_col).to_numpy()

    return X, y, w, d

In [10]:
def make_onehot_encoder():
    """Create a version-compatible OneHotEncoder."""
    try:
        # sklearn >= 1.2
        return OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse_output=True,
        )
    except TypeError:
        # older sklearn
        return OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse=True,
        )


# def build_ridge_pipeline(alpha=1.0):
#     """
#     Ridge pipeline with:
#       - numeric features: median imputation + standardization
#       - symbol_id: one-hot encoding
#     """
#     numeric_transformer = Pipeline(
#         steps=[
#             ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
#             ("scaler", StandardScaler()),
#         ]
#     )
#     categorical_transformer = make_onehot_encoder()
#     preprocessor = ColumnTransformer(
#         transformers=[
#             ("num", numeric_transformer, FEATURE_COLS),
#             ("sym", categorical_transformer, [SYMBOL_COL]),
#         ],
#         remainder="drop",
#     )
#     pipe = Pipeline(
#         steps=[
#             ("preprocessor", preprocessor),
#             ("ridge", Ridge(alpha=alpha, solver='lsqr')),
#         ]
#     )
#     return pipe

In [11]:
def build_ridge_pipeline(alpha=1.0):
    pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=False)),
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha, solver='lsqr')),
        ]
    )
    return pipe

In [12]:
def tune_ridge_alpha(
    alphas,
    train_partitions=range(0, 9),
    val_partition=9,
    train_sample_frac=0.35,
    seed=42,
):
    print("Loading sampled train partitions for tuning ...")
    train_df = load_partitions(
        train_partitions,
        sample_frac=train_sample_frac,
        seed=seed,
    )

    print("Loading full validation partition ...")
    val_df = load_partitions(
        [val_partition],
        sample_frac=1.0,
        seed=seed,
    )

    X_train, y_train, w_train, d_train = prepare_xywd(train_df)
    X_val, y_val, w_val, d_val = prepare_xywd(val_df)

    records = []

    for alpha in alphas:
        print(f"Training Ridge model with alpha={alpha} ...")
        model = build_ridge_pipeline(alpha=alpha)
        model.fit(X_train, y_train, ridge__sample_weight=w_train)

        pred_val = model.predict(X_val)
        val_r2 = official_weighted_r2(y_val, pred_val, w_val)

        records.append(
            {
                "alpha": alpha,
                "val_official_r2": val_r2,
                "val_rows": len(y_val),
            }
        )

    result_df = pd.DataFrame(records).sort_values(
        "val_official_r2", ascending=False
    ).reset_index(drop=True)
    
    best_alpha = result_df.loc[0, "alpha"]

    del train_df, val_df, X_train, y_train, w_train, d_train, X_val, y_val, w_val, d_val
    gc.collect()

    return result_df, best_alpha

In [11]:
alphas = [1e5, 5e5, 1e6, 5e6, 1e7]

tune_df, best_alpha = tune_ridge_alpha(
    alphas=alphas,
    train_partitions=range(0, 9),    # partitions 0..8
    val_partition=9,
    train_sample_frac=0.1,
    seed=DEFAULT_RANDOM_SEED,
)

display(tune_df)
print("Best alpha:", best_alpha)

Loading sampled train partitions for tuning ...
Loading full validation partition ...
Training Ridge model with alpha=100000.0 ...
Training Ridge model with alpha=500000.0 ...
Training Ridge model with alpha=1000000.0 ...
Training Ridge model with alpha=5000000.0 ...
Training Ridge model with alpha=10000000.0 ...


,alpha,val_official_r2,val_rows
0,5000000.0,0.002289,6274576
1,1000000.0,0.002125,6274576
2,10000000.0,0.002023,6274576
3,500000.0,0.001963,6274576
4,100000.0,0.001924,6274576


Best alpha: 5000000.0


In [ ]:
reference_rows = int(tune_df.loc[0, "val_rows"]) # 6274576
reference_alpha = tune_df.loc[0, "alpha"] # 5000000

In [13]:
def expanding_window_backtest(
    reference_alpha,
    reference_rows,
    start_val_partition=0,
    end_val_partition=8,
    train_sample_frac=0.1,
    pred_clip=None,
    seed=42,
):
    records = []

    for val_pid in range(start_val_partition, end_val_partition + 1):
        train_pids = list(range(val_pid))

        print("\n" + "=" * 80)
        print(f"Train partitions: {train_pids}")
        #print(f"Validation partition: {val_pid}")

        train_df = load_partitions(
            train_pids,
            sample_frac=train_sample_frac,
            seed=seed,
        )

        val_df = load_partitions(
            [val_pid],
            sample_frac=1,
            seed=seed,
        )

        X_train, y_train, w_train, d_train = prepare_xywd(train_df)
        X_val, y_val, w_val, d_val = prepare_xywd(val_df)

        current_rows = len(y_train)
        
        scaled_alpha = reference_alpha * (current_rows / reference_rows)
        
        print(f"Dynamic Alpha: {scaled_alpha:,.1f} "
              f"(scaled from {reference_alpha:,.1f} using {current_rows}/{reference_rows} rows)")
 
        model = build_ridge_pipeline(alpha=scaled_alpha)
        model.fit(X_train, y_train, ridge__sample_weight=w_train)

        pred_train = model.predict(X_train)
        pred_val = model.predict(X_val)

        if pred_clip is not None:
            pred_train = np.clip(pred_train, pred_clip[0], pred_clip[1])
            pred_val = np.clip(pred_val, pred_clip[0], pred_clip[1])

        train_r2 = official_weighted_r2(y_train, pred_train, w_train)
        val_r2 = official_weighted_r2(y_val, pred_val, w_val)

        records.append(
            {
                "train_partitions": str(train_pids),
                "val_partition": val_pid,
                "train_rows": current_rows,
                "val_rows": len(y_val),
                "alpha_used": scaled_alpha,  
                "train_official_r2": train_r2,
                "val_official_r2": val_r2,
                "train_sample_frac": train_sample_frac,
            }
        )

        # print(
        #     f"train_official_r2={train_r2:.6f} "
        #     f"val_official_r2={val_r2:.6f} "
        # )

        del train_df, val_df, X_train, y_train, w_train, d_train, X_val, y_val, w_val, d_val
        del pred_train, pred_val, model
        gc.collect()

    result_df = pd.DataFrame(records)
    return result_df

In [14]:
backtest_df = expanding_window_backtest(
    reference_alpha=5000000,   
    reference_rows=6274576,     
    start_val_partition=1,
    end_val_partition=9,    
    train_sample_frac=0.1,  
    pred_clip=DEFAULT_PRED_CLIP,
    seed=DEFAULT_RANDOM_SEED,
)

display(backtest_df)
print("Mean validation official R2:", backtest_df["val_official_r2"].mean())


Train partitions: [0]
Dynamic Alpha: 115,688.3 (scaled from 5,000,000.0 using 145179/6274576 rows)

Train partitions: [0, 1]
Dynamic Alpha: 300,383.6 (scaled from 5,000,000.0 using 376956/6274576 rows)

Train partitions: [0, 1, 2]
Dynamic Alpha: 510,787.5 (scaled from 5,000,000.0 using 640995/6274576 rows)

Train partitions: [0, 1, 2, 3]
Dynamic Alpha: 741,380.0 (scaled from 5,000,000.0 using 930369/6274576 rows)

Train partitions: [0, 1, 2, 3, 4]
Dynamic Alpha: 1,083,866.9 (scaled from 5,000,000.0 using 1360161/6274576 rows)

Train partitions: [0, 1, 2, 3, 4, 5]
Dynamic Alpha: 1,473,407.1 (scaled from 5,000,000.0 using 1849001/6274576 rows)

Train partitions: [0, 1, 2, 3, 4, 5, 6]
Dynamic Alpha: 1,866,804.2 (scaled from 5,000,000.0 using 2342681/6274576 rows)

Train partitions: [0, 1, 2, 3, 4, 5, 6, 7]
Dynamic Alpha: 2,260,201.3 (scaled from 5,000,000.0 using 2836361/6274576 rows)

Train partitions: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Dynamic Alpha: 2,651,284.3 (scaled from 5,000,000.0 using

,train_partitions,val_partition,train_rows,val_rows,alpha_used,train_official_r2,val_official_r2,train_sample_frac
0,[0],1,145179,2804247,1.156883e+05,0.013476,0.007330,0.1
1,"[0, 1]",2,376956,3036873,3.003836e+05,0.009967,0.006127,0.1
2,"[0, 1, 2]",3,640995,4016784,5.107875e+05,0.007902,0.004075,0.1
3,"[0, 1, 2, 3]",4,930369,5022952,7.413800e+05,0.006556,0.008649,0.1
4,"[0, 1, 2, 3, 4]",5,1360161,5348200,1.083867e+06,0.007813,0.006555,0.1
5,"[0, 1, 2, 3, 4, 5]",6,1849001,6203912,1.473407e+06,0.007556,0.004351,0.1
6,"[0, 1, 2, 3, 4, 5, 6]",7,2342681,6335560,1.866804e+06,0.006870,0.004299,0.1
7,"[0, 1, 2, 3, 4, 5, 6, 7]",8,2836361,6140024,2.260201e+06,0.006378,0.005874,0.1
8,"[0, 1, 2, 3, 4, 5, 6, 7, 8]",9,3327137,6274576,2.651284e+06,0.006389,0.002311,0.1


Mean validation official R2: 0.0055079207760808414
